In [1]:
import pandas as pd
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel
from torchvision import models, transforms
from PIL import Image
import os

# --- 1. Dataset Class to handle CSV and ZIP (Images) ---
class MultimodalDataset(torch.utils.data.Dataset):
    def __init__(self, csv_file, img_dir, tokenizer, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.tokenizer = tokenizer
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Text processing (LLM part)
        text = str(self.df.iloc[idx]['text'])
        label = self.df.iloc[idx]['label_idx']
        inputs = self.tokenizer(text, padding='max_length', truncation=True, max_length=128, return_tensors="pt")

        # Image processing (VLM part)
        img_id = self.df.iloc[idx]['id']
        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'image': image,
            'label': torch.tensor(label, dtype=torch.long)
        }

# --- 2. Multimodal Model (LLM + VLM Fusion) ---
class MisinformationModel(nn.Module):
    def __init__(self):
        super(MisinformationModel, self).__init__()
        # LLM Component
        self.bert = BertModel.from_pretrained('bert-base-uncased')

        # VLM/Vision Component
        self.resnet = models.resnet50(pretrained=True)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, 768) # Align with BERT size

        # Fusion Layer
        self.classifier = nn.Sequential(
            nn.Linear(768 + 768, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 2) # Binary: Rumor or Not
        )

    def forward(self, input_ids, attention_mask, image):
        text_features = self.bert(input_ids=input_ids, attention_mask=attention_mask).pooler_output
        image_features = self.resnet(image)

        # Combine Text (LLM) and Image (Vision)
        combined = torch.cat((text_features, image_features), dim=1)
        return self.classifier(combined)

# --- 3. Execution Setup ---
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])

# Initialize Model
model = MisinformationModel()
print("Multimodal Model (LLM + VLM Components) Initialized.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/mod

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 77.2MB/s]


Multimodal Model (LLM + VLM Components) Initialized.
